# RunnableWithMessageHistory – Detailed Notes

## Why do we need Memory?

By default, Large Language Models (LLMs) do **not remember** previous conversations. Every call to `invoke()` is treated as a new request. Therefore, we need a mechanism to store previous user and AI messages so the model can continue a conversation. This storage mechanism is called **Memory**.

---

## What is Memory?

Memory is a **container** that stores the conversation between the user and the AI. It keeps all chat messages together so they can be used in future interactions within the same session.

**Example:**

```text
Memory
│
├── Human : My name is Chidvilas.
├── AI    : Nice to meet you.
├── Human : I live in Hyderabad.
└── AI    : Great!
```

---

## What is History?

History refers to the **actual list of messages** stored inside the memory container. It contains Human, AI, and System messages that represent the conversation.

**Example:**

```text
History

Human : My name is Chidvilas.
AI    : Nice to meet you.
Human : I live in Hyderabad.
```

**Memory stores History.**

---

## Why do we need `MessagesPlaceholder`?

A `ChatPromptTemplate` normally contains only the System and Human messages. There is no location to insert previous conversations.

Without `MessagesPlaceholder`:

```text
System Message
↓

Current Human Message
```

The model never receives previous conversations.

By adding:

```python
MessagesPlaceholder(variable_name="history")
```

the prompt becomes:

```text
System Message
↓

Previous Chat History
↓

Current Human Message
```

Now LangChain automatically inserts all previous messages before the latest user message, enabling the model to maintain conversational context.

---

## Why do we create Memory separately?

LangChain does not assume where conversations should be stored. Different applications may choose different storage systems such as:

- In-Memory (RAM)
- Redis
- MongoDB
- PostgreSQL
- File Storage

Creating the memory object tells LangChain where chat history should be stored.

**Example:**

```python
memory = InMemoryChatMessageHistory()
```

---

## Why do we use `lambda session_id: memory`?

`RunnableWithMessageHistory` supports multiple users.

When a request arrives, LangChain asks:

> "Which memory belongs to this session?"

The function

```python
lambda session_id: memory
```

returns the appropriate memory object for the current session.

For a single-user application, returning the same memory object is sufficient.

For multiple users, each session should receive its own memory.

**Example:**

```python
store = {}

def get_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]
```

Now each user has an independent conversation history.

---

## Why do we use `history_messages_key="history"`?

`MessagesPlaceholder` has the variable name:

```python
MessagesPlaceholder(variable_name="history")
```

LangChain needs to know which placeholder should receive the stored messages.

Therefore,

```python
history_messages_key="history"
```

tells LangChain to insert all previous chat messages into the placeholder named `"history"`.

---

## Why do we use `input_messages_key="topic"`?

`input_messages_key` specifies which input represents the **new user message**.

For example, if the prompt is:

```python
Human:
Explain about {topic}
```

then the user's message comes from the variable `topic`.

Hence,

```python
input_messages_key="topic"
```

Variables such as `num` are only parameters used to format the response and are **not considered user chat messages**.

---

# Complete Workflow

```text
User
   │
   ▼
Current Input (topic)

   │
   ▼
RunnableWithMessageHistory

   │
   ├── Reads session_id
   │
   ├── Calls lambda(session_id)
   │
   ▼
Retrieves Memory

   │
   ▼
Gets Chat History

   │
   ▼
Inserts History into MessagesPlaceholder

   │
   ▼
Creates Final Prompt

System Message
↓

Previous Conversation
↓

Current User Message

↓

Language Model

↓

AI Response

↓

Stores New Conversation Back into Memory
```

---

# Key Points

- **Memory:** A container that stores chat conversations.
- **History:** The collection of previous Human, AI, and System messages stored inside memory.
- **MessagesPlaceholder:** A placeholder that tells LangChain where to insert previous conversation history.
- **`lambda session_id: memory`:** Returns the memory associated with the current session.
- **`history_messages_key`:** Connects stored history with the placeholder in the prompt.
- **`input_messages_key`:** Specifies which input variable represents the current user message.
- **`RunnableWithMessageHistory`:** Automatically loads previous messages before the model runs and saves the latest conversation after the response is generated.

In [3]:
!pip install -qU langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.5/70.5 kB 5.3 MB/s eta 0:00:00


In [1]:
from langchain_core.prompts import SystemMessagePromptTemplate, HumanMessagePromptTemplate, ChatPromptTemplate, MessagesPlaceholder

from langchain_core.runnables import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.output_parsers import StrOutputParser
from google.colab import userdata
import os

In [32]:
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI
import os
key= userdata.get('Gemini')
os.environ['GOOGLE_API_KEY']=key

In [33]:
Model=ChatGoogleGenerativeAI(
    model="gemini-2.5-flash")
parser=StrOutputParser()


In [34]:
memory=InMemoryChatMessageHistory()
history=InMemoryChatMessageHistory()

In [35]:
Systemp=" Any thin that user want to explain you give it only one single line"
human_msg=HumanMessagePromptTemplate.from_template(" Explain about {topic}")
Prompt_var=ChatPromptTemplate.from_messages(
    [Systemp,
     MessagesPlaceholder(variable_name='history'),
     human_msg])

In [36]:
chain= Prompt_var|Model|parser

In [17]:
Method=RunnableWithMessageHistory(
    chain,
    lambda sessionid:memory,
    history_messages_key='history',
    input_messages_key='topic'

)


In [ ]:
res= Method.invoke(
    {
     'topic':'Hello How are you'
    },
    config={
        "configurable": {
            "session_id": "user1"
        }
    }
)
print(res)

A common greeting, it initiates conversation and acknowledges someone's presence.
It politely inquires about their current well-being, fostering connection.


In [ ]:
res= Method.invoke(
    {
     'topic':'Iam chidvilas'
    },
    config={
        "configurable": {
            "session_id": "user1"
        }
    }
)
print(res)

It declares "I am the play of consciousness," asserting one's identity as pure, joyful awareness.
This signifies a spiritual realization that one's true nature is the manifesting delight of universal consciousness.


In [ ]:
res= Method.invoke(
    {
     'topic':'I am raghu, i do fishing for living'
    },
    config={
        "configurable": {
            "session_id": "user2"
        }
    }
)
print(res)

Raghu introduces himself directly, establishing his personal identity.
He then describes his profession as fishing, which provides his livelihood.


In [ ]:
res= Method.invoke(
    {
     'topic':'what is RAG'
    },
    config={
        "configurable": {
            "session_id": "user1"
        }
    }
)
print(res)

RAG (Retrieval Augmented Generation) is an AI technique for Large Language Models (LLMs) to provide more accurate, up-to-date answers.
It works by first retrieving relevant external data and then using it to augment the LLM's prompt before generating a response.


In [ ]:
res= Method.invoke(
    {
     'topic':'which place is heven for kids'
    },
    config={
        "configurable": {
            "session_id": "user2"
        }
    }
)
print(res)

"Heaven for kids" is less a specific location and more an environment where they feel completely safe, loved, and have boundless freedom to explore and imagine.
It's a space filled with laughter, adventure, and the magic of unburdened play, sparking endless joy and creativity.


In [37]:
memory.messages

[]

In [ ]:
###

WE are seperating the messages, as per users
- Msgs sent by by simmilar users grouped togeather

In [38]:

containter={}
def get_history(session_id):
  if session_id not in containter:
    containter[session_id]=InMemoryChatMessageHistory()
  return containter[session_id]

In [39]:
containter

{}

In [40]:
Method=RunnableWithMessageHistory(
    chain,
    get_history,
    input_messages_key="topic",
    history_messages_key="history",
)

In [41]:
get_history("user1")

InMemoryChatMessageHistory(messages=[])

In [42]:
res= Method.invoke(
    {
     'topic':'LLM FullForm'
    },
    config={
        "configurable": {
            "session_id": "user1"
        }
    }
)
print(res)

LLM stands for Large Language Model.


In [43]:
res= Method.invoke(
    {
     'topic':'EPITA Stands for'
    },
    config={
        "configurable": {
            "session_id": "user2"
        }
    }
)
print(res)

EPITA stands for École Pour l'Informatique et les Techniques Avancées (School for Computer Science and Advanced Techniques).


In [44]:
res= Method.invoke(
    {
     'topic':'ESILV fullform'
    },
    config={
        "configurable": {
            "session_id": "user1"
        }
    }
)
print(res)

ESILV stands for **École Supérieure d'Ingénieurs Léonard de Vinci**, a generalist engineering school in France.


In [45]:
res= Method.invoke(
    {
     'topic':'JP Morgon is a'
    },
    config={
        "configurable": {
            "session_id": "user1"
        }
    }
)
print(res)

J.P. Morgan is a leading global financial services firm and one of the largest investment banks in the world, offering services from investment banking to asset management.


In [47]:
containter

{'user1': InMemoryChatMessageHistory(messages=[HumanMessage(content='LLM FullForm', additional_kwargs={}, response_metadata={}), AIMessage(content='LLM stands for Large Language Model.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='ESILV fullform', additional_kwargs={}, response_metadata={}), AIMessage(content="ESILV stands for **École Supérieure d'Ingénieurs Léonard de Vinci**, a generalist engineering school in France.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='JP Morgon is a', additional_kwargs={}, response_metadata={}), AIMessage(content='J.P. Morgan is a leading global financial services firm and one of the largest investment banks in the world, offering services from investment banking to asset management.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]),
 'user2': InMemoryChatMessageHistory(messages=[HumanMessage(content

In [48]:
get_history("user1")

InMemoryChatMessageHistory(messages=[HumanMessage(content='LLM FullForm', additional_kwargs={}, response_metadata={}), AIMessage(content='LLM stands for Large Language Model.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='ESILV fullform', additional_kwargs={}, response_metadata={}), AIMessage(content="ESILV stands for **École Supérieure d'Ingénieurs Léonard de Vinci**, a generalist engineering school in France.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='JP Morgon is a', additional_kwargs={}, response_metadata={}), AIMessage(content='J.P. Morgan is a leading global financial services firm and one of the largest investment banks in the world, offering services from investment banking to asset management.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])])

In [49]:
get_history("user2")

InMemoryChatMessageHistory(messages=[HumanMessage(content='EPITA Stands for', additional_kwargs={}, response_metadata={}), AIMessage(content="EPITA stands for École Pour l'Informatique et les Techniques Avancées (School for Computer Science and Advanced Techniques).", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])])